# Exercise classifier: data, features, training, error analysis

Run from the **project root** (`jupyter lab` there, then open this file), or
execute the path cell below.

> **The shipped dataset is synthetic.** Anything you measure here describes the
> pipeline logic, not real-world accuracy. See the warning at the top of
> `src/synthetic.py`. Record real clips with
> `python -m src.main --record squat --exercise squat` before drawing conclusions.

In [ ]:
import os, sys
from pathlib import Path

# make sure we run from the project root so `src` and `configs` resolve
if not Path('src').is_dir():
    os.chdir('..')
sys.path.insert(0, str(Path.cwd()))
print('working directory:', Path.cwd())

import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=2, suppress=True)

## 1. Build the dataset

If `data/raw_sessions` is empty, generate the synthetic bench first:

```bash
python -m tools.generate_synthetic_data
```

In [ ]:
from src.session import load_sessions, summarise

sessions = load_sessions('data/raw_sessions')
print(summarise(sessions))
print()
print('synthetic:', all(s.meta.get('synthetic') for s in sessions))

## 2. Look at the raw signal before modelling anything

The rep signal for each exercise should show flat plateaus at rest and clean
excursions. If it does not, no classifier will save you - fix the recording
setup or the choice of primary metric first.

In [ ]:
from src.exercise_config import ExerciseLibrary

lib = ExerciseLibrary.from_dir('configs')
fig, axes = plt.subplots(len(lib), 1, figsize=(11, 2.1 * len(lib)), sharex=False)

for ax, name in zip(np.atleast_1d(axes), lib.names):
    cfg = lib[name]
    metric = cfg.primary_metric
    clip = next((s for s in sessions if s.label == name), None)
    if clip is None:
        ax.set_visible(False); continue
    v = clip.metric_series(metric)
    ax.plot(v, lw=1.2)
    ax.axhline(cfg.rep_counter.near_threshold, color='tab:green', ls='--', lw=1,
               label='near (rest)')
    ax.axhline(cfg.rep_counter.far_threshold, color='tab:red', ls='--', lw=1,
               label='far (worked)')
    ax.set_title(f'{cfg.display_name} - {metric} (azimuth {clip.meta.get("azimuth")} deg)',
                 fontsize=9)
    ax.set_ylabel('deg'); ax.legend(fontsize=7, loc='upper right')
plt.tight_layout(); plt.show()

## 3. Features

57 per-frame channels (27 engine metrics + 15 hip-centred, torso-normalised
landmark x/y pairs) reduced to 5 statistics over a 45-frame window -> 285 dims.

In [ ]:
from src.dataset import build_dataset, describe, grouped_split
from src.features import FRAME_FEATURE_NAMES, WINDOW_FEATURE_NAMES

print(f'{len(FRAME_FEATURE_NAMES)} per-frame -> {len(WINDOW_FEATURE_NAMES)} per-window')
ds = build_dataset(sessions, window=45, stride=10, only_good_form=True)
print(describe(ds))

### Why the split must be grouped

Consecutive windows from one clip overlap and are nearly identical. A random
split puts near-duplicates on both sides. The cell below quantifies the
inflation - it is usually 10-20 accuracy points of pure self-deception.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from src.train import make_estimator

# WRONG: random split, leaks overlapping windows
Xtr, Xte, ytr, yte = train_test_split(ds.X, ds.y, test_size=0.25, random_state=0,
                                      stratify=ds.y)
leaky = make_estimator('random_forest').fit(Xtr, ytr)
print(f'random split      : {accuracy_score(yte, leaky.predict(Xte)):.4f}   <- optimistic')

# RIGHT: whole clips held out
tr_idx, te_idx = grouped_split(ds, test_size=0.25, seed=0, group_by='clip')
tr, te = ds.subset(tr_idx), ds.subset(te_idx)
honest = make_estimator('random_forest').fit(tr.X, tr.y)
print(f'grouped by clip   : {accuracy_score(te.y, honest.predict(te.X)):.4f}   <- report this')

## 4. Compare models

Note the latency column: it is measured per **single window**, which is how
inference actually happens. A model that batches well can still be too slow here.

In [ ]:
import time
from sklearn.metrics import f1_score

rows = []
for kind in ['logreg', 'random_forest', 'svm']:
    pipe = make_estimator(kind)
    t0 = time.perf_counter(); pipe.fit(tr.X, tr.y); fit_s = time.perf_counter() - t0
    pred = pipe.predict(te.X)
    t0 = time.perf_counter()
    for i in range(min(200, len(te))):
        pipe.predict_proba(te.X[i:i + 1])
    ms = (time.perf_counter() - t0) / min(200, len(te)) * 1000
    rows.append((kind, accuracy_score(te.y, pred), f1_score(te.y, pred, average='macro'),
                 fit_s, ms))

print(f"{'model':<16}{'acc':>8}{'macroF1':>10}{'fit s':>8}{'ms/window':>12}")
for r in rows:
    print(f'{r[0]:<16}{r[1]:>8.4f}{r[2]:>10.4f}{r[3]:>8.1f}{r[4]:>12.2f}')
print('\nIf logreg already scores ~1.0, the task is close to linearly separable -')
print('on synthetic data that is expected, on real data it means check for leakage.')

## 5. Confusion matrix and feature importance

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

print(classification_report(te.y, honest.predict(te.X), zero_division=0))
fig, ax = plt.subplots(figsize=(5.5, 5))
ConfusionMatrixDisplay.from_predictions(te.y, honest.predict(te.X), ax=ax,
                                        xticks_rotation=45, colorbar=False)
ax.set_title('held-out, grouped by clip'); plt.tight_layout(); plt.show()

In [ ]:
imp = honest.named_steps['model'].feature_importances_
order = np.argsort(imp)[::-1][:22]
plt.figure(figsize=(8, 6))
plt.barh([WINDOW_FEATURE_NAMES[i] for i in order][::-1], imp[order][::-1])
plt.title('most informative window features'); plt.tight_layout(); plt.show()

## 6. Train and save the shipping bundle

Equivalent to `python -m src.train`. Saves the model, the IsolationForest
novelty gate, the feature names, the window size and the metrics.

In [ ]:
import argparse
from src.train import build_parser, train

args = build_parser().parse_args([])   # all defaults
bundle = train(args)
print('\nlabels:', bundle.labels)

## 7. Does it reject movements it has never seen?

This is the part that actually matters. A forest asked about an unknown movement
will happily return a known class with high confidence.

In [ ]:
from src.pipeline import MonitorPipeline

for s in load_sessions('data/test_unseen'):
    pipe = MonitorPipeline(lib, bundle=bundle)
    states = [st for st in pipe.replay(s) if st.detected][bundle.window:]
    if not states:
        continue
    unknown = sum(st.exercise is None for st in states) / len(states)
    guessed = {st.exercise for st in states if st.exercise}
    print(f'{s.label:<14} unknown {unknown:5.1%}   generic reps {pipe.generic_counter.count:>3}'
          f'   leaked labels: {sorted(guessed) or "-"}')

## 8. Error analysis: where does the pipeline actually fail?

Replays the fault test set end to end and lists every clip whose rep count is
wrong or whose expected error code was missed. Start debugging here.

In [ ]:
problems = []
for s in load_sessions('data/test_faults'):
    if s.label not in lib:
        continue
    pipe = MonitorPipeline(lib, bundle=bundle)
    pipe.replay(s)
    counted = pipe.rep_totals().get(s.label, 0)
    raised = set(pipe.detector.summary())
    missed = set(s.expected_codes) - raised
    if abs(counted - (s.expected_reps or 0)) > 1 or missed:
        problems.append((s.path.name, s.fault, s.expected_reps, counted,
                         sorted(missed), sorted(raised)))

print(f'{len(problems)} problem clips\n')
for name, fault, exp, got, missed, raised in problems:
    print(f'{name:<34} {fault:<12} reps {exp}->{got}  missed {missed}')
    print(f'{"":<34} raised {raised}')

## 9. Threshold calibration

How the shipped `max_speed` values were derived: measure good and faulty reps,
put the threshold between them. Do this for every threshold you change.

In [ ]:
from collections import defaultdict

speeds = defaultdict(list)
for s in load_sessions('data/test_faults'):
    if s.label not in lib:
        continue
    pipe = MonitorPipeline(lib, forced_exercise=s.label)
    pipe.replay(s)
    for r in pipe.counter_for(s.label).reps:
        speeds[(s.label, s.fault)].append(r.rom / r.duration)

for ex in lib.names:
    slow = np.array([x for (e, f), v in speeds.items() if e == ex and f != 'fast' for x in v])
    fast = np.array([x for (e, f), v in speeds.items() if e == ex and f == 'fast' for x in v])
    if slow.size and fast.size:
        print(f'{ex:<16} controlled {slow.min():4.0f}-{slow.max():4.0f}   '
              f'rushed {fast.min():4.0f}-{fast.max():4.0f}   '
              f'-> max_speed {(slow.max() + fast.min()) / 2:4.0f}  '
              f'(config: {lib[ex].tempo.max_speed})')

## 10. Full metrics report

Writes `docs/metrics_report.md` and `docs/metrics_report.json`, including the
MediaPipe benchmark and the end-to-end FPS estimate.

In [ ]:
from src.evaluate import main as evaluate_main
evaluate_main([])

---
## Next steps with real data

1. Record clips: `python -m src.main --record squat --exercise squat --subject alice --expected-reps 20`
2. Record bad-form clips into `data/test_faults` with `--fault shallow` etc.
3. **Include imperfect reps in the training set**, labelled with the correct
   exercise, plus a `standing_still` rest class with no config.
4. Retrain, then re-run sections 7-9. Recalibrate every `max_speed`.
5. Split by `subject`, not `clip`, once you have several people:
   `python -m src.train --group-by subject`. That is the number worth quoting.